In [ ]:
from networkx import Graph
from collections import defaultdict
import networkx as nx
import numpy as np

In [ ]:
# EX 1

G = Graph()
edges = defaultdict(int)

with open("ca-AstroPh.txt", "r") as f:
    for i, line in enumerate(f):
        if i >= 1500:
            break
        u, v = line.split()
        edges[(u, v)] += 1
        G.add_edge(int(u), int(v), weight=edges[(u, v)])

print(G)

In [ ]:
for node in G.nodes:
    egonet = G.subgraph(G.neighbors(node))
    
    Ni = egonet.number_of_nodes()
    Ei = egonet.number_of_edges()
    Wi = sum(data.get("weight", 1) for _, _, data in egonet.edges(data=True))
    
    A = nx.to_numpy_array(egonet, weight="weight")
    lambda_w = np.linalg.eigvals(A).real.max()
    
    features = {
        "Ni" : Ni,
        "Ei": Ei,
        "Wi": Wi,
        "lambda_w": lambda_w
    }
    nx.set_node_attributes(G, {node: features})
    


In [ ]:
from sklearn.linear_model import LinearRegression

nodes_list = [node for node in G.nodes if G.nodes[node]['Ni'] > 0 and G.nodes[node]['Ei'] > 0]

X = np.log(np.array([G.nodes[node]['Ni'] for node in nodes_list]))
Y = np.log(np.array([G.nodes[node]['Ei'] for node in nodes_list]))
len(X), len(Y)

In [ ]:
lg = LinearRegression()
lg.fit(X.reshape(-1, 1), Y)

In [ ]:
theta = lg.coef_[0]
C = np.exp(lg.intercept_)

In [ ]:
scores = {}
for i, node in enumerate(nodes_list):
    yi = np.exp(Y[i])
    xi = np.exp(X[i])
    
    pred = C * (xi ** theta)
    
    numerator = max(yi, pred)
    denominator = min(yi, pred)
        
    score = (numerator / denominator) * np.log(abs(yi - pred) + 1)
    scores[node] = score

In [ ]:
sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
top_10 = [n for n, s in sorted_scores[:10]]

top_10

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
colors = ['red' if n in top_10 else 'blue' for n in G.nodes()]
nx.draw(G, pos, node_color=colors, node_size=20)
plt.title("Top 10 Anomalies")
plt.show()

In [ ]:
lof_data = []
for i in range(len(nodes_list)):
    lof_data.append([Y[i], X[i]]) 
lof_data = np.array(lof_data)

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=30)
lof.fit(lof_data)

lof_score_values = -lof.negative_outlier_factor_

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

oddball_vals = np.array([scores[n] for n in nodes_list]).reshape(-1, 1)
oddball_norm = scaler.fit_transform(oddball_vals).flatten()

lof_norm = scaler.fit_transform(lof_score_values.reshape(-1, 1)).flatten()

combined_scores = {}
for i, node in enumerate(nodes_list):
    combined_scores[node] = oddball_norm[i] + lof_norm[i]

In [ ]:
sorted_combined = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
top_10_combined = [n for n, _ in sorted_combined[:10]]

plt.figure(figsize=(10, 8))
colors_combined = ['green' if n in top_10_combined else 'blue' for n in G.nodes()]
nx.draw(G, pos, node_color=colors_combined, node_size=20)
plt.title("Top 10 Anomalies")
plt.show()

In [ ]:
# EX 3
from scipy.io import loadmat
from torch_geometric.utils import from_scipy_sparse_matrix

data = loadmat('ACM.mat')

X = data['Attributes']
A = data['Network']
labels = data['Label'].flatten()

edge_index, _ = from_scipy_sparse_matrix(A)

In [ ]:
import torch

X = torch.FloatTensor(X.toarray())
A = torch.FloatTensor(A.toarray())


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Encoder(nn.Module):
    def __init__(self, input_dim):
        super(Encoder, self).__init__()
        self.conv1 = GCNConv(input_dim, 128)
        self.conv2 = GCNConv(128, 64)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return x

class AttributeDecoder(nn.Module):
    def __init__(self, input_dim):
        super(AttributeDecoder, self).__init__()
        self.conv1 = GCNConv(64, 128)
        self.conv2 = GCNConv(128, input_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class StructureDecoder(nn.Module):
    def __init__(self):
        super(StructureDecoder, self).__init__()
        self.conv1 = GCNConv(64, 64)

    def forward(self, z, edge_index):
        z = F.relu(self.conv1(z, edge_index))
       
        return torch.matmul(z, z.t())

class GAE(nn.Module):
    def __init__(self, input_dim):
        super(GAE, self).__init__()
        self.encoder = Encoder(input_dim)
        self.attr_decoder = AttributeDecoder(input_dim)
        self.struct_decoder = StructureDecoder()

    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        x_hat = self.attr_decoder(z, edge_index)
        a_hat = self.struct_decoder(z, edge_index)
        return x_hat, a_hat

In [ ]:
def loss_f(X, X_hat, A, A_hat, alpha):
    attr_loss = torch.sum((X - X_hat) ** 2)
    struct_loss = torch.sum((A - A_hat) ** 2)
    return alpha * attr_loss + (1 - alpha) * struct_loss

In [ ]:
model = GAE(X.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.004)
alpha = 0.8
epochs = 100

In [ ]:
from sklearn.metrics import roc_auc_score

print("Training...")


for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    X_hat, A_hat = model(X, edge_index)
    
    loss = loss_f(X, X_hat, A, A_hat, alpha)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
           
            attr_rec_error = torch.sum((X - X_hat) ** 2, dim=1)
            struct_rec_error = torch.sum((A - A_hat) ** 2, dim=1)
            
            scores = alpha * attr_rec_error + (1 - alpha) * struct_rec_error

            auc = roc_auc_score(labels, scores.numpy())
            print(f"Epoch {epoch} -- Loss = {loss.item()} -- ROC AUC = {auc}")
        